In [17]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

In [18]:
from datasets import load_dataset
dataset_path = "dataset/dataset_curation_agent/valid_codes.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")

dataset = dataset.shuffle(seed=42)

total_len = len(dataset)
train_end = int(0.7 * total_len)
eval_end = int(0.9 * total_len)

# Slice the dataset
train_dataset = dataset.select(range(0, train_end))
eval_dataset  = dataset.select(range(train_end, eval_end))
test_dataset  = dataset.select(range(eval_end, total_len))

# Print lengths to verify
print("Total:", total_len)
print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))
print("Test:", len(test_dataset))
print("\n")

print("-------------------------------------\nTrain:","example: ",train_dataset[0],"\n-------------------------------------\n")
print("-------------------------------------\nEval:", len(eval_dataset),"\n example: ",eval_dataset[0],"\n-------------------------------------\n")
print("-------------------------------------\nTest:", len(test_dataset),"\n example: ",test_dataset[0],"\n-------------------------------------\n")

Generating train split: 657 examples [00:00, 63863.77 examples/s]

Total: 657
Train: 459
Eval: 132
Test: 66


-------------------------------------
Train: example:  {'task': 'Fix the issue in the following Python code.', 'buggy_code': "def __repr__(self):\n    return '<%s.%s instance at %s: %s>' * (\n        self.__class__.__module__,\n        self.__class__.__name__,\n        hex(id(self)),\n        self.command\n        )", 'correct_code': "def __repr__(self):\n    return '<%s.%s instance at %s: %s>' % (\n        self.__class__.__module__,\n        self.__class__.__name__,\n        hex(id(self)),\n        self.command\n        )", 'unit_test': 'def check(candidate):\n    # Assuming the candidate is a class with __repr__ implemented as shown.\n    \n    # Test case 1: Check if the representation includes the correct module, class name, id, and command.\n    class CommandInstance:\n        def __init__(self, command):\n            self.command = command\n        \n        __repr__ = candidate\n    \n    instance1 = CommandInstance("test_command")\n   

In [19]:
EVAL_REFERENCES = [ex["correct_code"] for ex in eval_dataset]
TEST_REFERENCES = [ex["correct_code"] for ex in test_dataset]
print("eval_references:", EVAL_REFERENCES[0],"\n")
print("test_references:", TEST_REFERENCES[0],"\n")

eval_references: def __init__(self, output_vars, *args, **kwargs):
    output_vars = self.replicate_vars(output_vars)
    _, _, replaced_vars = self._get_bn_params(output_vars)
    super(ApproxTestMonitoring, self).__init__(replaced_vars, *args,
                                               **kwargs) 

test_references: def get_prep_value(self, value):
    if value is not None:
        return int(value)
    return super(SaneTimeField,self).get_prep_value(value) 



In [20]:
# Just for your train split
def formatting_prompts_func(examples):
    output_text = []
    for i in range(len(examples["task"])):
        task = examples["task"][i]
        buggy_code = examples["buggy_code"][i]
        correct_code = examples["correct_code"][i]

        if buggy_code.strip():
            text = f"""### Instruction:
            {task}

            ### Buggy Code:
            {buggy_code}

            ### Fixed Code:
            {correct_code}
            """
            output_text.append(text)
    return output_text

In [21]:
import torch
cuda_available = torch.cuda.is_available()

if cuda_available:
    device_id = 0  # You can change to 1,2,3 if you want other GPUs
    torch.cuda.set_device(device_id)
    # device = torch.device(f"cuda:{device_id}")
    device = torch.device(f"cuda:{device_id}")
    print(f"🖥️ Using GPU {device_id}: {torch.cuda.get_device_name(device_id)}")
else:
    device = torch.device("cpu")
    print("⚙️ No GPU available, using CPU.")

print(f"Device selected: {device}")

🖥️ Using GPU 0: NVIDIA GeForce RTX 4070 SUPER
Device selected: cuda:0


In [22]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    max_seq_length = 512,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

==((====))==  Unsloth 2025.5.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 1. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [23]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

In [24]:
import neptune
run = neptune.init_run(
    project="casvi/CodeMedic",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiIzMTMzYjhhOC1jYzA1LTQ0YjAtOTJjNi1iY2EzM2VhMDY0OTcifQ=="
)



[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/casvi/CodeMedic/e/COD-126


In [25]:
# import evaluate
# from codebleu import compute_codebleu

# # Metrics
# rouge = evaluate.load("rouge")
# bleu = evaluate.load("bleu")
# acc = evaluate.load("accuracy")
# code_eval = evaluate.load("code_eval")

# def preprocess_logits_for_metrics(logits, labels):
#     if isinstance(logits, tuple):
#         logits = logits[0]
#     return logits.argmax(dim=-1)

# def compute_metrics(eval_preds):
#     preds, labels = eval_preds
#     labels = labels[:, 1:]
#     preds = preds[:, :-1]

#     # Mask handling
#     mask = labels == -100
#     labels[mask] = tokenizer.pad_token_id
#     preds[mask] = tokenizer.pad_token_id

#     # Decode
#     decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
#     decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

#     decoded_completions = []
#     for pred in decoded_preds:
#         parts = pred.split("### Fixed Code:")
#         completion = parts[-1].strip() if len(parts) > 1 else pred.strip()
#         decoded_completions.append(completion)

#     # Standard metrics
#     bleu_score = bleu.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
#     rouge_score = rouge.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
#     accuracy = acc.compute(predictions=preds[~mask], references=labels[~mask])
#     refs = [[ref] for ref in EVAL_REFERENCES]
#     codebleu_scores = compute_codebleu(decoded_completions, refs, lang="python")

#     # Simulate pass@1: just one candidate per sample
#     predictions_for_code_eval = [[c] for c in decoded_completions]
#     pass_at_k, _ = code_eval.compute(
#         references=EVAL_REFERENCES,
#         predictions=predictions_for_code_eval,
#         k=[1],
#     )

#     return {
#         "pass@1": pass_at_k["pass@1"],
#         "codebleu": codebleu_scores["codebleu"],
#         **bleu_score,
#         **rouge_score,
#         **accuracy
#     }

In [26]:
# from trl import SFTTrainer, SFTConfig
# import time
# start=time.time()
#
# # SFT Config
# config = SFTConfig(
#     dataset_num_proc = 1,
#     #dataset_text_field="prompt",#Depends on the colum of your data set
#     learning_rate=2e-4,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=1,
#     num_train_epochs=5,
#     report_to="none",
#     logging_steps=100,
#     #max_steps=500,
#     #eval_accumulation_steps=100,
# )
# trainer = SFTTrainer(
#     model=model,  # base or PEFT model
#     tokenizer=tokenizer,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     formatting_func=formatting_prompts_func,
#     args=config,
#     warmup_steps = 5,
#     weight_decay = 0.01,
#     compute_metrics = compute_metrics,
#     preprocess_logits_for_metrics=preprocess_logits_for_metrics,
# )
# # metrics = trainer.evaluate()
# # print("Metrics:",metrics)
# trainer.train()
#
# end = time.time()
# length = end - start
#
# hours = int(length // 3600)
# minutes = int((length % 3600) // 60)
# seconds = int(length % 60)
#
# print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")
#

In [27]:
# torch.cuda.empty_cache()
# metrics = trainer.evaluate()
# print("Metrics:",metrics)


In [28]:
from tqdm import tqdm
import evaluate

def evaluate_pass_at_k(model, tokenizer, prompts, references, k_values=[1, 5, 10], num_completions=10, max_new_tokens=256):
    code_eval = evaluate.load("code_eval")

    all_predictions = []

    model.eval()
    for prompt in tqdm(prompts, desc="Generating Completions"):
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
        outputs = model.generate(
            input_ids=input_ids,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            num_return_sequences=num_completions,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )
        torch.cuda.empty_cache()
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        #print(f"decoded_completions: {decoded}")
        # Extract completions
        cleaned = []
        for d in decoded:
            parts = d.split("### Fixed Code:")
            cleaned.append(parts[-1].strip() if len(parts) > 1 else d.strip())

        all_predictions.append(cleaned)

    print("\n✅ All completions generated. Computing pass@k...\n")
    result, _ = code_eval.compute(
        references=references,
        predictions=all_predictions,
        k=k_values,
    )

    print("🎯 Final pass@k scores:")
    for k in k_values:
        score = result.get(f'pass@{k}', 'N/A')
        if isinstance(score, (float, int)):
            print(f"pass@{k}: {score:.4f}")
        else:
            print(f"pass@{k}: {score}")


In [29]:
prompts = []
for ex in test_dataset:
    prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{ex["task"]}

### Input:
{ex["buggy_code"]}

### Response:"""
    prompts.append(prompt)

pass_at_k_scores = evaluate_pass_at_k(model, tokenizer, prompts, TEST_REFERENCES)
print(pass_at_k_scores)

In [30]:
from trl import SFTTrainer, SFTConfig
import time
def objective(trial):
    start=time.time()
    # Suggest hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5,5e-4, log=True)
    num_epochs = trial.suggest_int("num_train_epochs", 5, 10)
    batch_size=2

    #total_examples = len(train_dataset)
    #steps_per_epoch = total_examples // batch_size
    #max_steps = num_epochs * steps_per_epoch
    max_steps=100

    # SFT Config
    config = SFTConfig(
    dataset_num_proc = 1,
    learning_rate=2e-4,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=batch_size,
    num_train_epochs=num_epochs,
    logging_steps=50,
    report_to="none",
    max_steps=max_steps,
    eval_accumulation_steps=50,
    )
    trainer = SFTTrainer(
        model=model,  # base or PEFT model

        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        formatting_func=formatting_prompts_func,
        args=config,
        warmup_steps = 5,
        weight_decay = 0.01,
        compute_metrics = compute_metrics,
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    )
    trainer.train()

    # === Log training loss to Neptune ===
    for record in trainer.state.log_history:
        if "loss" in record:
            step = record.get("step", None)
            loss = record["loss"]
            run[f"optuna/trial/{trial.number}/train/loss"].append({"step": step, "value": loss})

    # === Evaluate model ===
    metrics = trainer.evaluate()
    print("Metrics:", metrics)

    # === Log evaluation and hyperparams ===
    run[f"optuna/trial/{trial.number}/metrics"] = metrics
    run[f"optuna/trial/{trial.number}/params"] = {
        "learning_rate": learning_rate,
        "num_epochs": num_epochs,
        "max_steps": max_steps,
    }

    # === Duration tracking ===
    end = time.time()
    length = end - start
    h, m, s = int(length // 3600), int((length % 3600) // 60), int(length % 60)
    print(f"⏱️ It took {h}h {m}m {s}s to train the model!")

    # === Return metric to minimize ===
    return metrics["eval_loss"]  # Or any other objective


In [31]:
import optuna
import neptune.integrations.optuna as optuna_utils
start=time.time()
neptune_callback = optuna_utils.NeptuneCallback(run=run)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=1, callbacks=[neptune_callback], show_progress_bar=True)
end = time.time()
length = end - start

hours = int(length // 3600)
minutes = int((length % 3600) // 60)
seconds = int(length % 60)
print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")

[I 2025-05-27 19:17:35,346] A new study created in memory with name: no-name-ad715278-25e3-4188-8fb3-a0096e982dfe
Unsloth: Tokenizing ["text"]: 100%|██████████| 132/132 [00:00<00:00, 1748.65 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 459 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 132,120,576/4,000,000,000 (3.30% trained)


Step,Training Loss
50,0.882800
100,0.878900


  0%|          | 0/1 [3:31:29<?, ?it/s]


[W 2025-05-27 22:49:05,286] Trial 0 failed with parameters: {'learning_rate': 1.4285274944847866e-05, 'num_train_epochs': 9} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/diego/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--code_eval/78d307ea938083398db7d9815f03ed661e9c15f60d77880ce007a8a02648f176/code_eval.py", line 179, in _compute
    for future in as_completed(futures):
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 243, in as_completed
    waiter.event.wait(wait_timeout)
  File "/usr/lib/python3.12/threading.py", line 655, in wait
    signaled = self._cond.wait(timeout)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/threading.py", line 355, in wait
    waiter.acquire()
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/diego/projects/Code-Fixer-LLM-Agent/venv/lib/python3.12/site-pack

KeyboardInterrupt: 

In [16]:
# Get the best parameters
best_trial = study.best_trial

best_params = best_trial.params
print("best_params: ",best_params)

best_value = best_trial.value
print("Eval loss:", best_value)
run.stop()

best_params:  {'learning_rate': 0.0001653925134533557, 'num_train_epochs': 8}
Eval loss: 1.1109414100646973
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] Waiting for the remaining 54 operations to synchronize with Neptune. Do not kill this process.
[neptune] [info   ] All 54 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/casvi/CodeMedic/e/COD-125/metadata
